# 1 - import

In [1]:
import os
import json
import math
import random
from collections import Counter
from typing import List, Dict

import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.metrics import mean_squared_error

In [2]:
import csv
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

In [4]:
import csv
from datetime import datetime

# 2 - configs

In [5]:

CONFIG = {
    'annotations': r'E:\Balanced_20_Frames_Augmented\train_final.json',
    'data_root': r'E:\Balanced_20_Frames_Augmented\Train',
    'stats_file': r'E:\Balanced_20_Frames_Augmented\stats.json',
    'label_map': r'E:\Balanced_20_Frames_Augmented\label_map_final.json',  # optional
    'save_dir': './checkpoints_text2sign_2',
    'batch_size': 24,
    'epochs': 40,
    'lr': 1e-5,
    'patience': 5,
    'weight_decay': 1e-2,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'max_text_len': 32,
    'max_landmark_len': 70,
    'd_model': 512,
    'nhead': 8,
    'num_encoder_layers': 3,
    'num_decoder_layers': 3,
    'dropout': 0.1,
    'teacher_forcing_rate': 0.7,
    'save_every': 1,
    'seed': 42
}

os.makedirs(CONFIG['save_dir'], exist_ok=True)
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
random.seed(CONFIG['seed'])

# 3 - tokenizer

In [49]:
# class SimpleTokenizer:
#     def __init__(self, texts: List[str], min_freq: int = 1, max_vocab: int = None):
#         # word-level tokenizer
#         counter = Counter()
#         for t in texts:
#             tokens = self._tok(t)
#             counter.update(tokens)
#         # special tokens
#         self.pad_token = '<pad>'
#         self.sos_token = '<sos>'
#         self.eos_token = '<eos>'
#         self.unk_token = '<unk>'
#         specials = [self.pad_token, self.sos_token, self.eos_token, self.unk_token]
#         vocab_items = [w for w, c in counter.items() if c >= min_freq]
#         if max_vocab:
#             vocab_items = vocab_items[:max_vocab]
#         self.itos = specials + vocab_items
#         self.stoi = {w: i for i, w in enumerate(self.itos)}
#     def _tok(self, text: str):
#         # simple whitespace + lower
#         return text.lower().strip().split()
#     def encode(self, text: str, max_len: int):
#         toks = self._tok(text)
#         toks = toks[:max_len-2]  # reserve for sos/eos
#         ids = [self.stoi.get(t, self.stoi[self.unk_token]) for t in toks]
#         return [self.stoi[self.sos_token]] + ids + [self.stoi[self.eos_token]]
#     def decode(self, ids: List[int]):
#         words = []
#         for i in ids:
#             w = self.itos[i] if i < len(self.itos) else self.unk_token
#             if w in (self.sos_token, self.eos_token, self.pad_token):
#                 continue
#             words.append(w)
#         return " ".join(words)
#     @property
#     def vocab_size(self):
#         return len(self.itos)

In [50]:
class HFTokenizer:
    def __init__(self, texts, min_freq=1, max_vocab=None):
        # ---- Special tokens ----
        self.pad_token = "<pad>"
        self.sos_token = "<sos>"
        self.eos_token = "<eos>"
        self.unk_token = "<unk>"
        specials = [self.pad_token, self.sos_token, self.eos_token, self.unk_token]

        # ---- Tokenizer backend ----
        self.tokenizer = Tokenizer(WordPiece(unk_token=self.unk_token))
        self.tokenizer.pre_tokenizer = Whitespace()

        if max_vocab is None:
            max_vocab = 5000

        trainer = WordPieceTrainer(
            special_tokens=specials,
            min_frequency=min_freq,
            vocab_size=max_vocab,  
        )

        # ---- Train on provided texts ----
        self.tokenizer.train_from_iterator(texts, trainer)

        # ---- stoi / itos ----
        self.stoi = self.tokenizer.get_vocab()
        self.itos = sorted(self.stoi, key=lambda w: self.stoi[w])

    @property
    def vocab_size(self):
        return self.tokenizer.get_vocab_size()

    def encode(self, text, max_len):
        ids = self.tokenizer.encode(text.lower().strip()).ids
        ids = ids[:max_len - 2]
        return [self.stoi[self.sos_token]] + ids + [self.stoi[self.eos_token]]

    def decode(self, ids):
        out = []
        for i in ids:
            if i >= len(self.itos):
                continue
            tok = self.itos[i]
            if tok in (self.sos_token, self.eos_token, self.pad_token):
                continue
            out.append(tok)
        return " ".join(out)


# 4 - t2sDataset class

In [51]:
class TextToSignDataset(Dataset):
    def __init__(self, annotations_path, data_root, stats_path, tokenizer,
                 max_text_len=32, max_landmark_len=70, min_frames=5):

        with open(annotations_path, 'r') as f:
            self.annotations = json.load(f)

        with open(stats_path, 'r') as f:
            stats = json.load(f)

        mean = np.array(stats["spatial_mean"] + stats["temporal_mean"], dtype=np.float32)
        std = np.array(stats["spatial_std"] + stats["temporal_std"], dtype=np.float32)
        std[std < 1e-6] = 1.0

        self.mean = mean
        self.std = std
        self.data_root = data_root
        self.tokenizer = tokenizer
        self.max_text_len = max_text_len
        self.max_landmark_len = max_landmark_len
        self.min_frames = min_frames

        self.samples = []
        print("[Dataset] Scanning annotations...")
        for entry in tqdm(self.annotations):
            gloss = entry.get("gloss", "").strip()
            if not gloss:
                continue
            for inst in entry.get("instances", []):
                vid = inst.get("video_id")
                if not vid:
                    continue
                p = os.path.join(self.data_root, vid, "landmarks.json")
                if os.path.exists(p):
                    self.samples.append({"video_id": vid, "landmarks_path": p, "text": gloss})

        print(f"[Dataset] Total usable pairs: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def _extract_spatial_features(self, frame):
        if not isinstance(frame, dict):
            return None

        def safe_get(key, size):
            x = np.array(frame.get(key, []), dtype=np.float32).flatten()
            if len(x) > size:
                return x[:size]
            if len(x) < size:
                return np.pad(x, (0, size - len(x)))
            return x

        try:
            return np.concatenate([
                safe_get("pose", 132),
                safe_get("left_hand", 84),
                safe_get("right_hand", 84),
                safe_get("face", 1404),
                safe_get("left_hand_engineered", 19),
                safe_get("right_hand_engineered", 19),
            ])
        except:
            return None

    def __getitem__(self, idx):
        sample = self.samples[idx]
        path = sample["landmarks_path"]
        text = sample["text"]

        try:
            with open(path, "r") as f:
                frames = json.load(f)
            if not isinstance(frames, list) or len(frames) < self.min_frames:
                return None
        except:
            return None

        feats = [self._extract_spatial_features(fr) for fr in frames]
        if any(f is None for f in feats):
            return None

        spatial = np.array(feats, dtype=np.float32)
        if np.isnan(spatial).any() or np.isinf(spatial).any():
            return None

        temporal = np.diff(spatial, axis=0, prepend=spatial[:1])
        features = np.concatenate([spatial, temporal], axis=1)

        if features.shape[1] != self.mean.shape[0]:
            return None

        features = (features - self.mean) / self.std
        
        features = np.nan_to_num(
            features,
            nan=0.0,
            posinf=5.0,
            neginf=-5.0
        )
        features = np.clip(features, -10.0, 10.0)


        T, D = features.shape
        if T >= self.max_landmark_len:
            idxs = np.linspace(0, T - 1, self.max_landmark_len, dtype=int)
            features = features[idxs]
            lmask = np.ones(self.max_landmark_len, dtype=np.float32)
        else:
            pad = np.zeros((self.max_landmark_len - T, D), dtype=np.float32)
            features = np.concatenate([features, pad], axis=0)
            lmask = np.array([1]*T + [0]*(self.max_landmark_len - T), dtype=np.float32)

        tokens = self.tokenizer.encode(text, max_len=self.max_text_len)
        if len(tokens) < self.max_text_len:
            tokens += [self.tokenizer.stoi[self.tokenizer.pad_token]] * (self.max_text_len - len(tokens))
        else:
            tokens = tokens[:self.max_text_len]

        imask = [1 if t != self.tokenizer.stoi[self.tokenizer.pad_token] else 0 for t in tokens]

        return {
            "input_ids": torch.tensor(tokens, dtype=torch.long),
            "input_mask": torch.tensor(imask, dtype=torch.bool),
            "landmarks": torch.tensor(features, dtype=torch.float32),
            "landmark_mask": torch.tensor(lmask, dtype=torch.bool),
        }


def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None

    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "input_mask": torch.stack([b["input_mask"] for b in batch]),
        "landmarks": torch.stack([b["landmarks"] for b in batch]),
        "landmark_mask": torch.stack([b["landmark_mask"] for b in batch]),
    }


# 5 - t2s model

In [ ]:
def generate_square_subsequent_mask(sz: int, device):
    mask = torch.triu(torch.ones((sz, sz), device=device) * float('-inf'), diagonal=1)
    return mask  # shape (sz, sz)

class PositionalEmbedding(nn.Module):
    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        self.pos_emb = nn.Embedding(max_len, d_model)
    def forward(self, seq_len):
        # returns (seq_len, d_model)
        positions = torch.arange(seq_len, device=self.pos_emb.weight.device)
        return self.pos_emb(positions)  # (seq_len, d_model)

class TextToSignModel(nn.Module):
    def __init__(self, vocab_size, landmark_dim, cfg: Dict):
        super().__init__()
        d_model = cfg['d_model']
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.landmark_dim = landmark_dim
        self.max_landmark_len = cfg['max_landmark_len']
        self.max_text_len = cfg['max_text_len']

        # text encoder
        # print("[Model] Initializing text encoder...")
        self.tok_embed = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.text_pos = PositionalEmbedding(self.max_text_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=cfg['nhead'],
                                                   dim_feedforward=d_model*4, dropout=cfg['dropout'],
                                                   batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=cfg['num_encoder_layers'])

        # print("[Model] Building landmark projection...")
        self.landmark_in_proj = nn.Linear(landmark_dim, d_model)
        self.start_frame = nn.Parameter(torch.randn(1, d_model))  # (1, d_model)
        self.landmark_pos = PositionalEmbedding(self.max_landmark_len, d_model)

        # transformer decoder
        # print("[Model] Initializing decoder...")
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=cfg['nhead'],
                                                   dim_feedforward=d_model*4, dropout=cfg['dropout'],
                                                   batch_first=True, activation='gelu')
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=cfg['num_decoder_layers'])

        # output projection
        self.out_proj = nn.Linear(d_model, landmark_dim)

        # print("[Model] Applying weight initialization...")
        self._init_weights()
        # print("[Model] Initialization complete.\n")

    def _init_weights(self):
        for n, p in self.named_parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode_text(self, input_ids, input_mask):
        # print("[Model] Encoding text...")
        emb = self.tok_embed(input_ids)  # (B, T_text, d_model)
        pos = self.text_pos(emb.shape[1]).unsqueeze(0)  # (1, T_text, d_model)
        src = emb + pos  # (B, T_text, d_model)
        src_key_padding_mask = ~input_mask  # True = padding
        memory = self.encoder(src, src_key_padding_mask=src_key_padding_mask)  # (B, T_text, d_model)
        # print("[Model] Text encoding complete.")
        return memory, src_key_padding_mask

    def forward(self, input_ids, input_mask, tgt_landmarks=None, teacher_forcing_ratio=0.9):
        """
        Text → Landmark regression
        Inputs:
            input_ids       : (B, T_text)
            input_mask      : (B, T_text)
            tgt_landmarks   : (B, T_landmark, landmark_dim) | optional
        Output:
            preds           : (B, T_landmark, landmark_dim)
        """

        B = input_ids.size(0)
        device = input_ids.device

        memory, src_key_padding_mask = self.encode_text(input_ids, input_mask)

        T = self.max_landmark_len
        d = self.d_model

        # ---- initial decoder input ----
        prev = self.start_frame.unsqueeze(0).expand(B, 1, d)  # (B,1,d)

        outputs = []
        pos = self.landmark_pos(T).unsqueeze(0)  # (1,T,d)

        for t in range(T):
            tgt = torch.cat(outputs + [prev], dim=1) if outputs else prev
            cur_len = tgt.size(1)

            tgt = tgt + pos[:, :cur_len, :]
            tgt_mask = generate_square_subsequent_mask(cur_len, device)

            dec_out = self.decoder(
                tgt,
                memory,
                tgt_mask=tgt_mask,
                memory_key_padding_mask=src_key_padding_mask
            )

            last_feat = dec_out[:, -1:, :]       # (B,1,d)
            pred_frame = self.out_proj(last_feat)  # (B,1,landmark_dim)

            outputs.append(last_feat)

            # ---- teacher forcing ----
            if tgt_landmarks is not None and random.random() < teacher_forcing_ratio:
                prev = self.landmark_in_proj(tgt_landmarks[:, t:t+1, :])
            else:
                prev = self.landmark_in_proj(pred_frame)

        dec_feats = torch.cat(outputs, dim=1)     # (B,T,d)
        preds = self.out_proj(dec_feats)         # (B,T,landmark_dim)

        return preds


# 5.2 - Loss and Vel

In [53]:
class MaskedMSELoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, preds, targets, mask):
        # mask: (B, T) or (B, T, 1)
        if mask.dim() == preds.dim() - 1:
            mask = mask.unsqueeze(-1)

        mask = mask.float()

        diff = (preds - targets) ** 2
        diff = diff * mask

        denom = mask.sum().clamp(min=1.0)
        return diff.sum() / denom


In [54]:
class MaskedSmoothL1Loss(nn.Module):
    def __init__(self, beta=1.0):
        super().__init__()
        self.beta = beta

    def forward(self, pred, target, mask):
        # pred/target: (B,T,D)
        # mask: (B,T)
        loss = F.smooth_l1_loss(
            pred, target,
            reduction='none',
            beta=self.beta
        )  # (B,T,D)

        loss = loss.mean(dim=-1)   # (B,T)
        loss = loss * mask
        return loss.sum() / (mask.sum() + 1e-6)


In [55]:
def temporal_smoothness_loss(pred, mask):
    # pred: (B,T,D)
    # mask: (B,T)
    vel = pred[:, 1:] - pred[:, :-1]      # (B,T-1,D)
    vel_mask = mask[:, 1:]

    loss = vel.abs().mean(dim=-1)          # (B,T-1)
    loss = loss * vel_mask
    return loss.sum() / (vel_mask.sum() + 1e-6)


In [56]:
def velocity_loss(x, mask):
    v = x[:,1:] - x[:,:-1]
    m = mask[:,1:]
    return (v.abs().mean(dim=-1) * m).sum() / (m.sum() + 1e-6)


# 5.3 - Metrics

In [57]:

# metrics_dir = os.path.join(CONFIG['save_dir'], "metrics")
# os.makedirs(metrics_dir, exist_ok=True)

# metrics_csv = os.path.join(metrics_dir, "epoch_metrics.csv")

# if not os.path.exists(metrics_csv):
#     with open(metrics_csv, "w", newline="") as f:
#         writer = csv.writer(f)
#         writer.writerow([
#             "epoch",
#             "train_loss",
#             "val_loss",
#             "val_pose_loss",
#             "val_velocity_loss",
#             "val_mpjpe",
#             "val_velocity_error",
#             "teacher_forcing_ratio",
#             "lr"
#         ])


In [58]:
def mpjpe(pred, gt, mask):
    # pred, gt: (B,T,D)
    # mask: (B,T)
    diff = pred - gt
    dist = torch.norm(diff, dim=-1)  # (B,T)
    dist = dist * mask
    return dist.sum() / (mask.sum() + 1e-6)

def velocity_error(pred, gt, mask):
    v_pred = pred[:,1:] - pred[:,:-1]
    v_gt   = gt[:,1:]   - gt[:,:-1]
    m = mask[:,1:]

    err = torch.norm(v_pred - v_gt, dim=-1)
    err = err * m
    return err.sum() / (m.sum() + 1e-6)



In [ ]:
def plot_metrics(csv_path, out_dir):
    df = pd.read_csv(csv_path)
    epochs = df["epoch"]

    plots = {
        "loss": ["train_loss", "val_loss"],
        "pose_vs_smooth": ["val_pose_loss", "val_velocity_loss"],
        "mpjpe": ["val_mpjpe"],
        "velocity_error": ["val_velocity_error"],
        "teacher_forcing": ["teacher_forcing_ratio"]
    }

    for name, cols in plots.items():
        plt.figure(figsize=(8,5))
        for c in cols:
            plt.plot(epochs, df[c], label=c)
        plt.xlabel("Epoch")
        plt.ylabel(name)
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"{name}.png"))
        plt.close()

# 6 - training and eval

In [60]:
def landmark_accuracy(y_pred, y_true, mask, alpha=5.0):
    """
    y_pred, y_true: [frames, dim]
    mask: [frames] boolean mask for valid frames
    """

    valid = mask.bool()
    if valid.sum() == 0:
        return None

    y_pred = y_pred[valid]
    y_true = y_true[valid]

    # reshape to [frames, keypoints, 2]
    F, D = y_pred.shape
    K = D // 2
    y_pred = y_pred.reshape(F, K, 2)
    y_true = y_true.reshape(F, K, 2)

    dist = torch.norm(y_pred - y_true, dim=-1)  # [frames, keypoints]
    mean_dist = dist.mean()   # scalar

    acc = torch.exp(-mean_dist / alpha)
    return acc.item()


In [61]:
def train_one_epoch(model, dataloader, optimizer, criterion, cfg, device, tf_ratio):
    print("\n[Train] Starting epoch...")
    model.train()
    total_loss = 0.0
    count = 0
    
    scaler = torch.cuda.amp.GradScaler(
        enabled=(device.type == "cuda")
    )

    for batch in tqdm(dataloader, desc="Train", leave=False):
        if batch is None:
            continue

        input_ids = batch['input_ids'].to(device)
        input_mask = batch['input_mask'].to(device)
        landmarks = batch['landmarks'].to(device)
        landmark_mask = batch['landmark_mask'].to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            preds = model(
                input_ids,
                input_mask,
                tgt_landmarks=landmarks,
                teacher_forcing_ratio=tf_ratio
            )

            pose_loss = criterion(preds, landmarks, landmark_mask)
            smooth_loss = temporal_smoothness_loss(preds, landmark_mask)
            loss = pose_loss + 0.1 * smooth_loss

        # backward (scaled)
        scaler.scale(loss).backward()

        # unscale before clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # optimizer step
        scaler.step(optimizer)
        scaler.update()


        total_loss += loss.item()
        count += 1

    avg_loss = total_loss / max(1, count)
    print(f"[Train] Epoch complete. Avg Loss: {avg_loss:.6f}")
    return avg_loss

@torch.no_grad()
def validate(model, dataloader, criterion, cfg, device, epoch=None):
    print("\n[Val] Running validation...")
    model.eval()

    total_loss = 0.0
    total_pose = 0.0
    total_smooth = 0.0
    total_mpjpe = 0.0
    total_vel_err = 0.0
    count = 0

    for batch in tqdm(dataloader, desc="Val", leave=False):
        if batch is None:
            continue

        input_ids = batch['input_ids'].to(device)
        input_mask = batch['input_mask'].to(device)
        landmarks = batch['landmarks'].to(device)
        landmark_mask = batch['landmark_mask'].to(device)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            preds = model(
                input_ids,
                input_mask,
                tgt_landmarks=None,
                teacher_forcing_ratio=0.0
            )

            pose_loss = criterion(preds, landmarks, landmark_mask)
            smooth_loss = temporal_smoothness_loss(preds, landmark_mask)
            loss = pose_loss + 0.1 * smooth_loss

        total_loss += loss.item()
        total_pose += pose_loss.item()
        total_smooth += smooth_loss.item()

        total_mpjpe += mpjpe(preds, landmarks, landmark_mask).item()
        total_vel_err += velocity_error(preds, landmarks, landmark_mask).item()

        count += 1

    metrics = {
        "val_loss": total_loss / max(1, count),
        "val_pose_loss": total_pose / max(1, count),
        "val_velocity_loss": total_smooth / max(1, count),
        "val_mpjpe": total_mpjpe / max(1, count),
        "val_velocity_error": total_vel_err / max(1, count),
    }

    print(
        f"[Val] Loss: {metrics['val_loss']:.6f} | "
        f"Pose: {metrics['val_pose_loss']:.6f} | "
        f"Smooth: {metrics['val_velocity_loss']:.6f} | "
        f"MPJPE: {metrics['val_mpjpe']:.4f} | "
        f"VelErr: {metrics['val_velocity_error']:.4f}"
    )

    return metrics


In [62]:
@torch.no_grad()
def generate_from_text(model, tokenizer: HFTokenizer, text: str, cfg, device):
    model.eval()
    enc = tokenizer.encode(text, max_len=cfg['max_text_len'])
    if len(enc) < cfg['max_text_len']:
        enc = enc + [tokenizer.stoi[tokenizer.pad_token]] * (cfg['max_text_len'] - len(enc))
    input_ids = torch.tensor([enc], dtype=torch.long).to(device)
    input_mask = torch.tensor([[1 if i != tokenizer.stoi[tokenizer.pad_token] else 0 for i in enc]], dtype=torch.bool).to(device)
    preds = model(input_ids, input_mask, tgt_landmarks=None, teacher_forcing_ratio=0.0)  # (1, T, D)
    preds = preds.cpu().numpy()[0]  # (T, D)
    return preds


# 7 - main

In [63]:
cfg_init = CONFIG

In [64]:
log_path = os.path.join(cfg_init['save_dir'], "training_log.csv")

log_path = os.path.join(cfg_init['save_dir'], "training_log.csv")

if not os.path.exists(log_path):
    with open(log_path, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "val_loss", "val_mse", "val_acc", "timestamp"])


In [ ]:

def main(cfg):
    device = torch.device(cfg['device'])
    print(f"[Run] device: {device}")
    
    
    # -------------------------- metric dir-------------
    os.makedirs(cfg['save_dir'], exist_ok=True)
    metrics_dir = os.path.join(cfg['save_dir'], "metrics")
    os.makedirs(metrics_dir, exist_ok=True)

    metrics_csv = os.path.join(metrics_dir, "epoch_metrics.csv")

    if not os.path.exists(metrics_csv):
        with open(metrics_csv, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                "epoch",
                "train_loss",
                "val_loss",
                "val_pose_loss",
                "val_velocity_loss",
                "val_mpjpe",
                "val_velocity_error",
                "teacher_forcing_ratio",
                "lr"
            ])
    # --------------------------------------------------


    # 1) tokenization of glosses
    with open(cfg['annotations'], 'r') as f:
        ann = json.load(f)
    texts = [e.get('gloss', '').strip() for e in ann if e.get('gloss', '').strip()]
    tokenizer = HFTokenizer(texts, min_freq=1)
    print(f"[Vocab] size: {tokenizer.vocab_size}")

    # 2) build dataset
    dataset = TextToSignDataset(cfg['annotations'], cfg['data_root'], cfg['stats_file'],
                                tokenizer, max_text_len=cfg['max_text_len'],
                                max_landmark_len=cfg['max_landmark_len'])
    # quick split
    val_frac = 0.15
    n_val = max(1, int(len(dataset)*val_frac))
    n_train = len(dataset) - n_val
    train_ds, val_ds = random_split(dataset, [n_train, n_val])
    print(f"[Split] train={len(train_ds)}, val={len(val_ds)}")

    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True,
                              collate_fn=collate_fn, num_workers=0, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=cfg['batch_size'], shuffle=False,
                            collate_fn=collate_fn, num_workers=0)

    # determine landmark dim by sampling one item
    sample_item = None
    for i in range(len(dataset)):
        s = dataset[i]
        if s is not None:
            sample_item = s; break
    if sample_item is None:
        raise RuntimeError("No valid samples found.")
    landmark_dim = sample_item['landmarks'].shape[1]
    print(f"[Landmark dim] {landmark_dim}")

    # 3) model
    model = TextToSignModel(vocab_size=tokenizer.vocab_size, landmark_dim=landmark_dim, cfg=cfg).to(device)
    print(f"[Model] params: {sum(p.numel() for p in model.parameters()):,}")

    # 4) optimizer / scheduler / criterion
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    # criterion = MaskedMSELoss()
    criterion = MaskedSmoothL1Loss(beta=1.0)
    patience = 5
    epochs_no_improve = 0
    
    # scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    best_val_loss = float('inf')
    for epoch in range(cfg['epochs']):
        print(f"\n=== Epoch {epoch+1}/{cfg['epochs']} ===")
        tf_ratio = max(
            0.2,
            cfg['teacher_forcing_rate'] * (1 - epoch / cfg['epochs'])
        )
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, cfg, device, tf_ratio)
        # val_loss, val_mse, val_acc = validate(model, val_loader, criterion, cfg, device)
        # print(f"Train loss: {train_loss:.6f} | Val loss: {val_loss:.6f} | Val MSE (per sample): {val_mse} | Val Acc : {val_acc}")

        val_metrics = validate(model, val_loader, criterion, cfg, device, epoch)

        with open(metrics_csv, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                epoch + 1,
                train_loss,
                val_metrics["val_loss"],
                val_metrics["val_pose_loss"],
                val_metrics["val_velocity_loss"],
                val_metrics["val_mpjpe"],
                val_metrics["val_velocity_error"],
                tf_ratio,
                optimizer.param_groups[0]['lr']
            ])

        # if (epoch + 1) % 5 == 0:
        #     plot_metrics(metrics_csv, metrics_dir)


        plot_metrics(metrics_csv, metrics_dir)

        
        # save checkpoint
        if (epoch+1) % cfg['save_every'] == 0:
            path = os.path.join(cfg['save_dir'], f'text2sign_epoch_{epoch+1}.pth')
            torch.save({
                'epoch': epoch+1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'tokenizer': tokenizer.itos,
                'cfg': cfg
            }, path)
            print(f"[Saved] {path}")

        # early stop
        if val_metrics["val_loss"] < best_val_loss:
            best_val_loss = val_metrics["val_loss"]

            epochs_no_improve = 0
            torch.save({
                'epoch': epoch+1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'tokenizer': tokenizer.itos,
                'cfg': cfg
            }, os.path.join(cfg['save_dir'], 'best_text2sign.pth'))
            print("[Saved] best_text2sign.pth")
        else:
            epochs_no_improve += 1
            print(f"[Early Stop] No improvement for {epochs_no_improve}/{patience} epochs.")
            
        if epochs_no_improve == patience:
            print(f"== Early stopping triggered after {epoch+1} epochs (patience={patience}). ==")
            break

    # Example generation on first few validation samples
    print("\n[Generation examples]")
    model.eval()
    for i in range(min(5, len(val_ds))):
        sample = val_ds[i]
        if sample is None: continue
        text = tokenizer.decode(sample['input_ids'].numpy().tolist())
        preds = generate_from_text(model, tokenizer, text, cfg, device)
        # Un-normalize preds back to original space
        with open(cfg['stats_file'], 'r') as f:
            stats = json.load(f)
        mean = np.array(stats.get('spatial_mean', []) + stats.get('temporal_mean', []), dtype=np.float32)
        std = np.array(stats.get('spatial_std', []) + stats.get('temporal_std', []), dtype=np.float32)
        std[std < 1e-6] = 1.0
        preds_unnorm = preds * std + mean
        print(f"Text: {text}")
        print(f"Generated frames shape: {preds_unnorm.shape}")
        # Save prediction array
        save_arr = preds_unnorm.astype(np.float32)
        np.save(os.path.join(cfg['save_dir'], f"pred_{i}.npy"), save_arr)

        # Also save CSV
        np.savetxt(
            os.path.join(cfg['save_dir'], f"pred_{i}.csv"),
            save_arr.reshape(save_arr.shape[0], -1),
            delimiter=","
        )

        print(f"Saved pred_{i}.npy and pred_{i}.csv")
        print(f"Text: {text} | Frames: {preds_unnorm.shape[0]}")



In [66]:
main(CONFIG)

[Run] device: cuda
[Vocab] size: 576
[Dataset] Scanning annotations...


100%|██████████| 200/200 [00:00<00:00, 1666.00it/s]

[Dataset] Total usable pairs: 10000
[Split] train=8500, val=1500


[Landmark dim] 3484
[Model] params: 25,988,508

=== Epoch 1/40 ===

[Train] Starting epoch...


[Train] Epoch complete. Avg Loss: 0.401961

[Val] Running validation...


...

[Val] Loss: 0.326410 | Pose: 0.325358 | Smooth: 0.010521 | MPJPE: 50.8843 | VelErr: 52.2436
[Saved] ./checkpoints_text2sign_2\text2sign_epoch_39.pth
[Early Stop] No improvement for 1/5 epochs.

=== Epoch 40/40 ===

[Train] Starting epoch...


[Train] Epoch complete. Avg Loss: 0.247905

[Val] Running validation...


[Val] Loss: 0.328153 | Pose: 0.327208 | Smooth: 0.009452 | MPJPE: 50.7485 | VelErr: 52.2291
[Saved] ./checkpoints_text2sign_2\text2sign_epoch_40.pth
[Early Stop] No improvement for 2/5 epochs.

[Generation examples]


c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\torch\nn\modules\transformer.py:408: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ..\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)


Text: demand
Generated frames shape: (70, 3484)
Saved pred_4.npy and pred_4.csv
Text: demand | Frames: 70


# 8 - Testing

In [70]:
import torch, json, os, numpy as np
from tokenizers.models import WordPiece

# -----------------------
# Load checkpoint
# -----------------------
ckpt_path = "checkpoints_text2sign_2/best_text2sign.pth"
checkpoint = torch.load(ckpt_path, map_location="cpu")

cfg = checkpoint["cfg"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"[Run] device: {device}")

# -----------------------
# Restore tokenizer (MATCH TRAINING SCRIPT)
# -----------------------
tokenizer = HFTokenizer(texts=[], min_freq=1)   # dummy init

tokenizer.itos = checkpoint["tokenizer"]
tokenizer.stoi = {w: i for i, w in enumerate(tokenizer.itos)}

tokenizer.tokenizer.model = WordPiece(
    vocab=tokenizer.stoi,
    unk_token=tokenizer.unk_token
)

print(f"[Tokenizer] vocab restored: {tokenizer.vocab_size}")

# -----------------------
# Recover landmark_dim SAFELY
# -----------------------
state_dict = checkpoint["model_state_dict"]
landmark_dim = state_dict["out_proj.weight"].shape[0]

print(f"[Landmark dim] {landmark_dim}")

# -----------------------
# Build model
# -----------------------
model = TextToSignModel(
    vocab_size=tokenizer.vocab_size,
    landmark_dim=landmark_dim,
    cfg=cfg
).to(device)

model.load_state_dict(state_dict)
model.eval()

print("==> Model loaded successfully")

# -----------------------
# Load normalization stats
# -----------------------
with open(cfg["stats_file"], "r") as f:
    stats = json.load(f)

mean = np.array(
    stats["spatial_mean"] + stats["temporal_mean"],
    dtype=np.float32
)
std = np.array(
    stats["spatial_std"] + stats["temporal_std"],
    dtype=np.float32
)
std[std < 1e-6] = 1.0

# -----------------------
# Inference
# -----------------------
os.makedirs(cfg["save_dir"], exist_ok=True)

texts_to_generate = [
    "hello",
    "thank you",
    "how are you",
]

with torch.no_grad():
    for i, text in enumerate(texts_to_generate):
        preds = generate_from_text(
            model,
            tokenizer,
            text,
            cfg,
            device
        )  # (T, D)

        preds_unnorm = preds * std + mean

        np.save(
            os.path.join(cfg["save_dir"], f"pred_{i}.npy"),
            preds_unnorm.astype(np.float32)
        )

        np.savetxt(
            os.path.join(cfg["save_dir"], f"pred_{i}.csv"),
            preds_unnorm.reshape(preds_unnorm.shape[0], -1),
            delimiter=","
        )

        print(f"[OK] '{text}' → {preds_unnorm.shape} frames saved")


[Run] device: cuda
[Tokenizer] vocab restored: 576
[Landmark dim] 3484
==> Model loaded successfully
[OK] 'hello' → (70, 3484) frames saved
[OK] 'thank you' → (70, 3484) frames saved
[OK] 'how are you' → (70, 3484) frames saved


In [75]:
import numpy as np
import cv2
import mediapipe as mp
import os

# -----------------------------
# Config
# -----------------------------
INPUT_FILE = os.path.join(cfg['save_dir'], "pred_1.npy")
OUTPUT_VIDEO = "pred_1_output.mp4"
FPS = 15
IMG_SIZE = 800  

POSE = 33
FACE = 468
HAND = 21

POSE_DIM = POSE * 4
HAND_DIM = HAND * 4
FACE_DIM = FACE * 3

SPATIAL_DIM = POSE_DIM + HAND_DIM * 2 + FACE_DIM  # 1742

# -----------------------------
# MediaPipe connections
# -----------------------------
mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands

POSE_CONN = mp_pose.POSE_CONNECTIONS
HAND_CONN = mp_hands.HAND_CONNECTIONS

# -----------------------------
# Utilities
# -----------------------------
def clip_and_scale(xy):
    """Clip to [0,1] and scale to image size"""
    xy = np.nan_to_num(xy)
    xy = np.clip(xy, 0.0, 1.0)
    return (xy * IMG_SIZE).astype(np.int32)

def safe_line(img, p1, p2, color, thickness, max_dist=120):
    """Draw line only if points are close enough"""
    if np.linalg.norm(p1 - p2) < max_dist:
        cv2.line(img, tuple(p1), tuple(p2), color, thickness)

def split_frame(vec):
    """Split spatial-only landmarks (IGNORE temporal part)"""

    spatial = vec[:SPATIAL_DIM]

    i = 0
    pose = spatial[i:i+POSE_DIM].reshape(POSE, 4)
    i += POSE_DIM

    left = spatial[i:i+HAND_DIM].reshape(HAND, 4)
    i += HAND_DIM

    right = spatial[i:i+HAND_DIM].reshape(HAND, 4)
    i += HAND_DIM

    face = spatial[i:i+FACE_DIM].reshape(FACE, 3)

    return pose, left, right, face

# -----------------------------
# Load prediction
# -----------------------------
data = np.load(INPUT_FILE)   # (T, D)
frames = data.shape[0]

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    fourcc,
    FPS,
    (IMG_SIZE, IMG_SIZE)
)

print("Rendering video...")

# -----------------------------
# Render loop
# -----------------------------
for t in range(frames):
    frame = np.ones((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8) * 255

    pose_raw, left_raw, right_raw, _ = split_frame(data[t])

    # Pose
    pose_xy = clip_and_scale(pose_raw[:, :2])
    pose_vis = pose_raw[:, 3] > 0.5   # visibility channel

    for a, b in POSE_CONN:
        if pose_vis[a] and pose_vis[b]:
            safe_line(frame, pose_xy[a], pose_xy[b], (0, 0, 255), 2)

    for i, p in enumerate(pose_xy):
        if pose_vis[i]:
            cv2.circle(frame, tuple(p), 3, (0, 0, 200), -1)

    # Hands (no visibility channel → distance-filter only)
    left_xy = clip_and_scale(left_raw[:, :2])
    right_xy = clip_and_scale(right_raw[:, :2])

    for a, b in HAND_CONN:
        safe_line(frame, left_xy[a], left_xy[b], (255, 0, 0), 2)
        safe_line(frame, right_xy[a], right_xy[b], (0, 255, 0), 2)

    for p in left_xy:
        cv2.circle(frame, tuple(p), 3, (200, 0, 0), -1)
    for p in right_xy:
        cv2.circle(frame, tuple(p), 3, (0, 200, 0), -1)

    writer.write(frame)

writer.release()
print(f"== Video saved to {OUTPUT_VIDEO}")


Rendering video...
== Video saved to pred_1_output.mp4
